<a href="https://colab.research.google.com/github/haris444/autonomous-agents/blob/dev/eval_personality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Per-Agent Personality vs Survival Analysis

Runs `eval_robust` (50 episodes, greedy policy) on **all 11 experiment checkpoints** sequentially.
For each experiment, extracts per-agent: return, survival%, attack%, coop%, give%.
Then analyzes: which behavioral traits correlate with survival in each environment?

| Exp | Name | Source |
|---|---|---|
| A | Baseline | **Hardcoded** (local checkpoint, already evaluated) |
| C | No Predator | Drive: `ab_test_full/C_no_predator/` |
| D-local | No Social (hierarchy only) | **Hardcoded** (local checkpoint, already evaluated) |
| B2 | No Ledger | Drive: `ab_test_r2/B_no_ledger/` |
| E | No Rich Food | Drive: `ab_test_r2/E_no_rich_food/` |
| F | No Ledger + No Social | Drive: `ab_test_r2/F_no_ledger_no_social/` |
| G | 3 Predators | Drive: `ab_test_r2/G_3_predators/` |
| H | Strong Social | Drive: `strong_social/sac_h_strong_social/` |
| H-NoPred | Strong Social, No Pred | Drive: `strong_social/sac_h_strong_social_no_pred/` |
| H-3Pred | Strong Social, 3 Pred | Drive: `strong_social/sac_h_strong_social_3pred/` |
| H-NoRich | Strong Social, No Rich | Drive: `strong_social/sac_h_strong_social_no_rich/` |

**A and D-local** were evaluated locally (50 episodes, greedy). Data hardcoded from `docs/ABLATION_RESULTS.md`.

**Runtime:** ~10-15 min on T4 GPU (9 checkpoints x 50 episodes x 128 steps, sequential)

## 1. Setup

In [ ]:
!git clone -b dev https://github.com/haris444/autonomous-agents.git
%cd autonomous-agents
!pip install -q pyyaml

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Mount Drive & Locate Checkpoints

In [ ]:
from google.colab import drive
import os, glob

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/AutonomousAgents'

# ── Checkpoints on Google Drive (9 experiments) ──
EXPERIMENTS = {
    'C: No Predator':       f'{DRIVE_BASE}/ab_test_full/C_no_predator/',
    'B2: No Ledger':        f'{DRIVE_BASE}/ab_test_r2/B_no_ledger/',
    'E: No Rich Food':      f'{DRIVE_BASE}/ab_test_r2/E_no_rich_food/',
    'F: No Ledger+Social':  f'{DRIVE_BASE}/ab_test_r2/F_no_ledger_no_social/',
    'G: 3 Predators':       f'{DRIVE_BASE}/ab_test_r2/G_3_predators/',
    'H: Strong Social':     f'{DRIVE_BASE}/strong_social/sac_h_strong_social/',
    'H-NoPred':             f'{DRIVE_BASE}/strong_social/sac_h_strong_social_no_pred/',
    'H-3Pred':              f'{DRIVE_BASE}/strong_social/sac_h_strong_social_3pred/',
    'H-NoRich':             f'{DRIVE_BASE}/strong_social/sac_h_strong_social_no_rich/',
}

def find_checkpoint(folder):
    """Find best checkpoint in folder: prefer checkpoint_final.pt, else latest checkpoint_*.pt"""
    final = os.path.join(folder, 'checkpoint_final.pt')
    if os.path.exists(final):
        return final
    ckpts = sorted(glob.glob(os.path.join(folder, 'checkpoint_*.pt')))
    return ckpts[-1] if ckpts else None

# Resolve checkpoint paths
found = {}
print(f'{"Experiment":<25} {"Found":>5}  Checkpoint')
print('-' * 90)
for name, folder in EXPERIMENTS.items():
    ckpt = find_checkpoint(folder) if os.path.isdir(folder) else None
    status = 'YES' if ckpt else 'NO'
    print(f'{name:<25} {status:>5}  {ckpt or folder}')
    if ckpt:
        found[name] = ckpt

print(f'\n{len(found)}/{len(EXPERIMENTS)} Drive checkpoints found')

# ── Hardcoded results for A and D-local (evaluated locally, 50 episodes greedy) ──
# Source: docs/ABLATION_RESULTS.md — Per-Agent Profiles
HARDCODED = {
    'A: Baseline': [
        {'agent': 0, 'return': 335.8, 'survival': 80.0, 'attack': 5.7, 'coop': 21.1, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 1, 'return': 344.2, 'survival': 80.0, 'attack': 3.2, 'coop': 22.1, 'give': 12.4, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 2, 'return': 375.5, 'survival': 82.0, 'attack': 2.7, 'coop': 25.5, 'give': 11.7, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 3, 'return': 393.5, 'survival': 86.0, 'attack': 2.3, 'coop': 26.5, 'give': 4.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 4, 'return': 416.5, 'survival': 86.0, 'attack': 1.2, 'coop': 27.1, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 5, 'return': 428.3, 'survival': 88.0, 'attack': 2.0, 'coop': 27.8, 'give': 1.5, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 6, 'return': 508.9, 'survival': 94.0, 'attack': 1.1, 'coop': 30.3, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 7, 'return': 497.5, 'survival': 88.0, 'attack': 1.1, 'coop': 28.0, 'give': 0.2, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
    ],
    'D-local: No Social': [
        {'agent': 0, 'return': 392.2, 'survival': 82.0, 'attack': 4.3, 'coop': 23.4, 'give': 0.5, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 1, 'return': 425.1, 'survival': 84.0, 'attack': 2.8, 'coop': 25.7, 'give': 10.9, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 2, 'return': 450.5, 'survival': 88.0, 'attack': 2.4, 'coop': 27.2, 'give': 10.0, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 3, 'return': 477.0, 'survival': 92.0, 'attack': 2.2, 'coop': 28.5, 'give': 3.2, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 4, 'return': 492.1, 'survival': 94.0, 'attack': 1.5, 'coop': 29.6, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 5, 'return': 506.5, 'survival': 96.0, 'attack': 1.7, 'coop': 29.8, 'give': 1.4, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 6, 'return': 566.0, 'survival': 98.0, 'attack': 1.0, 'coop': 30.9, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
        {'agent': 7, 'return': 529.4, 'survival': 96.0, 'attack': 1.4, 'coop': 28.3, 'give': 0.1, 'move': 0.0, 'dmg_out': 0.0, 'dmg_in': 0.0, 'food_given': 0.0},
    ],
}

print(f'\nHardcoded: {list(HARDCODED.keys())} (already evaluated locally)')
print(f'Total experiments: {len(found) + len(HARDCODED)}')

## 3. Evaluation Function

In [ ]:
import numpy as np
from collections import Counter
from env.environment import GridWorld
from analysis.utils import load_sac, get_device

def eval_per_agent(checkpoint_path, n_episodes=50):
    """Run eval and return structured per-agent data."""
    device = get_device()
    sac, config, ckpt = load_sac(checkpoint_path, device)
    n = config.n_agents
    print(f'  Loaded ({n} agents). Running {n_episodes} episodes...')

    agent_stats = {i: {
        'returns': [], 'survived': [], 'action_counts': [],
        'damage_dealt': [], 'damage_taken': [],
        'food_given': [], 'coop_count': [],
    } for i in range(n)}

    for ep in range(n_episodes):
        env = GridWorld(config, device=device)
        obs = env.reset()
        ac = {i: Counter() for i in range(n)}
        tr = {i: 0.0 for i in range(n)}

        for step in range(config.max_steps_per_episode):
            obs_batched = {k: v.unsqueeze(0) for k, v in obs.items()}
            with torch.no_grad():
                directions, action_types = sac.get_actions(obs_batched, deterministic=True)
            dirs = directions.squeeze(0)
            acts = action_types.squeeze(0)
            for i in range(n):
                ac[i][acts[i].item()] += 1
            obs, rewards, dones, info = env.step(dirs, acts)
            for i in range(n):
                tr[i] += rewards[i].item()
            if dones.all():
                break

        alive = env.agent_alive.cpu().numpy()
        ledger = env.ledger.tensor.cpu().numpy()

        for i in range(n):
            agent_stats[i]['returns'].append(tr[i])
            agent_stats[i]['survived'].append(bool(alive[i]))
            agent_stats[i]['action_counts'].append(dict(ac[i]))
            agent_stats[i]['damage_dealt'].append(float(ledger[i, :, 0].sum()))
            agent_stats[i]['damage_taken'].append(float(ledger[:, i, 0].sum()))
            agent_stats[i]['food_given'].append(float(ledger[i, :, 1].sum()))
            agent_stats[i]['coop_count'].append(float(ledger[i, :, 2].sum()))

        if (ep + 1) % 10 == 0:
            avg_ret = np.mean([np.mean(agent_stats[i]['returns']) for i in range(n)])
            print(f'    Episode {ep+1}/{n_episodes} (avg return: {avg_ret:+.1f})')

    # Build per-agent summary
    results = []
    for i in range(n):
        s = agent_stats[i]
        move_pcts, atk_pcts, give_pcts, coop_pcts = [], [], [], []
        for ac_dict in s['action_counts']:
            t = max(sum(ac_dict.values()), 1)
            move_pcts.append(ac_dict.get(0, 0) / t * 100)
            atk_pcts.append(ac_dict.get(1, 0) / t * 100)
            give_pcts.append(ac_dict.get(2, 0) / t * 100)
            coop_pcts.append(ac_dict.get(4, 0) / t * 100)

        results.append({
            'agent': i,
            'return': np.mean(s['returns']),
            'survival': np.mean(s['survived']) * 100,
            'attack': np.mean(atk_pcts),
            'coop': np.mean(coop_pcts),
            'give': np.mean(give_pcts),
            'move': np.mean(move_pcts),
            'dmg_out': np.mean(s['damage_dealt']),
            'dmg_in': np.mean(s['damage_taken']),
            'food_given': np.mean(s['food_given']),
        })

    return results

## 4. Run All Evaluations (Sequential)

In [ ]:
import time

# Start with hardcoded A and D-local
all_results = dict(HARDCODED)
print(f'Pre-loaded {len(all_results)} hardcoded experiments: {list(all_results.keys())}')

# Run eval on Drive checkpoints
for i, (name, path) in enumerate(found.items()):
    print(f'\n[{i+1}/{len(found)}] {name}')
    t0 = time.time()
    try:
        all_results[name] = eval_per_agent(path, n_episodes=50)
        elapsed = time.time() - t0
        avg_surv = np.mean([r['survival'] for r in all_results[name]])
        print(f'  Done in {elapsed:.0f}s — avg survival: {avg_surv:.0f}%')
    except Exception as e:
        print(f'  FAILED: {e}')

print(f'\nCompleted {len(all_results)} total experiments ({len(HARDCODED)} hardcoded + {len(all_results) - len(HARDCODED)} evaluated)')

## 5. Per-Agent Personality Tables

In [ ]:
import pandas as pd

# Build master dataframe
rows = []
for exp_name, agents in all_results.items():
    for a in agents:
        rows.append({
            'experiment': exp_name,
            'agent': f'A{a["agent"]}',
            'return': round(a['return'], 1),
            'survival': round(a['survival'], 1),
            'attack': round(a['attack'], 1),
            'coop': round(a['coop'], 1),
            'give': round(a['give'], 1),
            'move': round(a['move'], 1),
            'dmg_out': round(a['dmg_out'], 1),
            'food_given': round(a['food_given'], 1),
        })

df = pd.DataFrame(rows)

# Print per-experiment tables
for exp_name in all_results:
    exp_df = df[df['experiment'] == exp_name].sort_values('survival', ascending=False)
    print(f'\n{"=" * 90}')
    print(f'{exp_name}')
    print(f'{"=" * 90}')
    print(exp_df[['agent', 'return', 'survival', 'attack', 'coop', 'give']].to_string(index=False))

    # Diversity
    surv_range = exp_df['survival'].max() - exp_df['survival'].min()
    surv_std = exp_df['survival'].std()
    print(f'  Survival range: {surv_range:.0f}pp  std: {surv_std:.1f}')

## 6. Personality vs Survival — Per Experiment

In [ ]:
import matplotlib.pyplot as plt

# Scatter: attack% vs survival% for each experiment
experiments = list(all_results.keys())
n_exp = len(experiments)
cols = 3
rows_plot = (n_exp + cols - 1) // cols

fig, axes = plt.subplots(rows_plot, cols, figsize=(5 * cols, 4 * rows_plot))
fig.suptitle('Attack% vs Survival% — Per Agent, Per Experiment', fontsize=14, fontweight='bold')

for idx, exp_name in enumerate(experiments):
    ax = axes.flat[idx]
    exp_df = df[df['experiment'] == exp_name]

    ax.scatter(exp_df['attack'], exp_df['survival'], s=80, c='#d62728', alpha=0.8, edgecolors='black', linewidths=0.5)

    # Label each point with agent ID
    for _, row in exp_df.iterrows():
        ax.annotate(row['agent'], (row['attack'], row['survival']),
                    fontsize=7, ha='center', va='bottom', xytext=(0, 4),
                    textcoords='offset points')

    ax.set_xlabel('Attack %')
    ax.set_ylabel('Survival %')
    ax.set_title(exp_name, fontsize=10)
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(n_exp, len(axes.flat)):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig('personality_attack_vs_survival.png', dpi=150)
plt.show()

In [ ]:
# Scatter: coop% vs survival% for each experiment
fig, axes = plt.subplots(rows_plot, cols, figsize=(5 * cols, 4 * rows_plot))
fig.suptitle('Coop% vs Survival% — Per Agent, Per Experiment', fontsize=14, fontweight='bold')

for idx, exp_name in enumerate(experiments):
    ax = axes.flat[idx]
    exp_df = df[df['experiment'] == exp_name]

    ax.scatter(exp_df['coop'], exp_df['survival'], s=80, c='#1f77b4', alpha=0.8, edgecolors='black', linewidths=0.5)

    for _, row in exp_df.iterrows():
        ax.annotate(row['agent'], (row['coop'], row['survival']),
                    fontsize=7, ha='center', va='bottom', xytext=(0, 4),
                    textcoords='offset points')

    ax.set_xlabel('Coop %')
    ax.set_ylabel('Survival %')
    ax.set_title(exp_name, fontsize=10)
    ax.grid(True, alpha=0.3)

for idx in range(n_exp, len(axes.flat)):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig('personality_coop_vs_survival.png', dpi=150)
plt.show()

In [ ]:
# Scatter: give% vs survival% for each experiment
fig, axes = plt.subplots(rows_plot, cols, figsize=(5 * cols, 4 * rows_plot))
fig.suptitle('Give% vs Survival% — Per Agent, Per Experiment', fontsize=14, fontweight='bold')

for idx, exp_name in enumerate(experiments):
    ax = axes.flat[idx]
    exp_df = df[df['experiment'] == exp_name]

    ax.scatter(exp_df['give'], exp_df['survival'], s=80, c='#2ca02c', alpha=0.8, edgecolors='black', linewidths=0.5)

    for _, row in exp_df.iterrows():
        ax.annotate(row['agent'], (row['give'], row['survival']),
                    fontsize=7, ha='center', va='bottom', xytext=(0, 4),
                    textcoords='offset points')

    ax.set_xlabel('Give %')
    ax.set_ylabel('Survival %')
    ax.set_title(exp_name, fontsize=10)
    ax.grid(True, alpha=0.3)

for idx in range(n_exp, len(axes.flat)):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig('personality_give_vs_survival.png', dpi=150)
plt.show()

## 7. Correlation Summary

In [ ]:
# Per-experiment correlation between behavioral traits and survival
print(f'{"Experiment":<25} {"Atk↔Surv":>10} {"Coop↔Surv":>10} {"Give↔Surv":>10} {"Surv Range":>10} {"Surv Std":>10}')
print('-' * 80)

corr_rows = []
for exp_name in all_results:
    exp_df = df[df['experiment'] == exp_name]

    surv_range = exp_df['survival'].max() - exp_df['survival'].min()
    surv_std = exp_df['survival'].std()

    # Pearson correlation (only meaningful if there's variance)
    if surv_std > 1:
        atk_corr = exp_df['attack'].corr(exp_df['survival'])
        coop_corr = exp_df['coop'].corr(exp_df['survival'])
        give_corr = exp_df['give'].corr(exp_df['survival'])
    else:
        atk_corr = coop_corr = give_corr = float('nan')

    print(f'{exp_name:<25} {atk_corr:>10.2f} {coop_corr:>10.2f} {give_corr:>10.2f} {surv_range:>9.0f}pp {surv_std:>9.1f}')

    corr_rows.append({
        'experiment': exp_name,
        'atk_surv_corr': atk_corr,
        'coop_surv_corr': coop_corr,
        'give_surv_corr': give_corr,
        'surv_range': surv_range,
        'surv_std': surv_std,
    })

corr_df = pd.DataFrame(corr_rows)

print(f'\n--- Interpretation ---')
print('Negative atk↔surv = less aggression -> more survival')
print('Positive coop↔surv = more cooperation -> more survival')
print('NaN = no variance in survival (all 0% or all 100%)')

## 8. Diversity Across Experiments

In [ ]:
# Bar chart: survival range (max-min) per experiment
fig, ax = plt.subplots(figsize=(12, 5))

exp_names = corr_df['experiment'].tolist()
ranges = corr_df['surv_range'].tolist()

colors = ['#d62728' if r > 40 else '#ff7f0e' if r > 15 else '#2ca02c' for r in ranges]
bars = ax.bar(range(len(exp_names)), ranges, color=colors, edgecolor='black', linewidth=0.5)

ax.set_xticks(range(len(exp_names)))
ax.set_xticklabels(exp_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Survival Range (max - min agent)')
ax.set_title('Agent Personality Divergence by Experiment', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, ranges):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}pp', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('personality_divergence_by_experiment.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 5))

corr_data = corr_df[['atk_surv_corr', 'coop_surv_corr', 'give_surv_corr']].values
im = ax.imshow(corr_data.T, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(exp_names)))
ax.set_xticklabels(exp_names, rotation=45, ha='right', fontsize=9)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Attack ↔ Survival', 'Coop ↔ Survival', 'Give ↔ Survival'])
ax.set_title('Behavioral Trait ↔ Survival Correlation by Experiment', fontweight='bold')

# Annotate cells
for i in range(3):
    for j in range(len(exp_names)):
        val = corr_data[j, i]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
plt.savefig('personality_correlation_heatmap.png', dpi=150)
plt.show()

## 9. Save Results

In [ ]:
import json, shutil

# Save to Drive
drive_out = f'{DRIVE_BASE}/personality_analysis'
os.makedirs(drive_out, exist_ok=True)

# Save master CSV
df.to_csv('personality_all_agents.csv', index=False)
shutil.copy2('personality_all_agents.csv', drive_out)

# Save correlation summary
corr_df.to_csv('personality_correlations.csv', index=False)
shutil.copy2('personality_correlations.csv', drive_out)

# Save raw results as JSON
with open('personality_raw.json', 'w') as f:
    json.dump(all_results, f, indent=2)
shutil.copy2('personality_raw.json', drive_out)

# Save plots
for png in ['personality_attack_vs_survival.png', 'personality_coop_vs_survival.png',
            'personality_give_vs_survival.png', 'personality_divergence_by_experiment.png',
            'personality_correlation_heatmap.png']:
    if os.path.exists(png):
        shutil.copy2(png, drive_out)

print(f'Saved to: {drive_out}')
print(f'Files: {os.listdir(drive_out)}')